# 2.3 章节实践

## 一、客观题

1. `functional_pass`、`performance_valid`、`performance_beneficial` 分别回答什么问题？
2. Path A 与 Path B 是否只改变“是否融合”？还改变了什么？
3. Path B 比 Path A 慢时，功能实验能否通过？
4. 为什么输出目录必须尚不存在？
5. 判断：旧记录中的 4.28% 可以直接填写为当前结果。


## 二、简单编程题：预检清单

从环境变量读取一次性工作副本、模型、tokenizer 和新输出目录，验证路径、精确基线、工作树、旧二进制和 Device，并生成不含敏感环境变量的预检字典。


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess

BASELINE = "60c6371cd30894d9896dfa979b86c6f892b6cbda"
CANDIDATE_SHA256 = "3dbe660a5933584fc9d434100ea4dc5c1600186a494b46df96da6aea55612126"
required = ("MUDUOXINYU_ROOT", "MUDUOXINYU_MODEL", "MUDUOXINYU_TOKENIZER", "MUDUOXINYU_OUTPUT")
missing = [name for name in required if not os.environ.get(name, "").strip()]
if missing:
    raise RuntimeError(f"缺少环境变量：{missing}")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


chapter_root = Path(os.environ["MUDUOXINYU_ROOT"]).expanduser().resolve()
chapter_model = Path(os.environ["MUDUOXINYU_MODEL"]).expanduser().resolve()
chapter_tokenizer = Path(os.environ["MUDUOXINYU_TOKENIZER"]).expanduser().resolve()
chapter_output = Path(os.environ["MUDUOXINYU_OUTPUT"]).expanduser().resolve()
assert chapter_root.is_dir() and chapter_model.is_file() and chapter_tokenizer.is_file()
assert not chapter_output.exists(), "章节实践必须使用新的输出目录"
head = subprocess.check_output(["git", "-C", str(chapter_root), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(
    ["git", "-C", str(chapter_root), "status", "--porcelain", "--untracked-files=normal"], text=True
).splitlines()
candidate = chapter_root / "src/backend/npuBackend.cpp"
binary = chapter_root / "muduoXinyu"
candidate_sha256 = file_sha256(candidate) if candidate.is_file() else None
subprocess.run(["npu-smi", "info"], check=True)
preflight = {
    "root": str(chapter_root),
    "head": head,
    "baseline_head": head == BASELINE,
    "candidate_sha256": candidate_sha256,
    "candidate_verified": candidate_sha256 == CANDIDATE_SHA256,
    "binary_present": binary.is_file(),
    "worktree_status": status,
    "model": {"bytes": chapter_model.stat().st_size, "sha256": file_sha256(chapter_model)},
    "tokenizer": {"bytes": chapter_tokenizer.stat().st_size, "sha256": file_sha256(chapter_tokenizer)},
    "output_new": True,
}
assert preflight["baseline_head"] and preflight["candidate_verified"] and preflight["binary_present"]
print(json.dumps(preflight, ensure_ascii=False, indent=2))


## 三、中等编程题：独立 A/B

确保 `MUDUOXINYU_ROOT` 已按 1.2 安全应用补丁并构建，然后用章节实践专用的新输出目录运行一次 A/B。检查 marker、fallback=0、token exact、调用次数、sample 1~4、三轮 mean/CV，以及 `run_manifest.json` 中的执行顺序和资产哈希。


In [ ]:
SRC = Path("src").resolve()
subprocess.run([
    "bash", str(SRC / "run_ab_benchmark.sh"),
    "--muduo-root", str(chapter_root),
    "--model", str(chapter_model),
    "--tokenizer", str(chapter_tokenizer),
    "--output-dir", str(chapter_output),
], check=True)
chapter_result = json.loads((chapter_output / "result.json").read_text(encoding="utf-8"))
chapter_manifest = json.loads((chapter_output / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(chapter_result["verdict"], ensure_ascii=False, indent=2))
print("execution_order:", chapter_manifest["execution_order"])


## 四、困难编程题：复合变量解释

结合调用次数、mean、CV、dtype/Cast、布局与 launch 开销，分别回答：功能是否通过、数据是否有效、性能是否受益。然后设计一个后续单变量实验，例如只固定 dtype 或只改变运行顺序；不要预设 FlashAttention 更快，也不要修改原始数据迎合结论。


In [ ]:
verdict = chapter_result["verdict"]
perf = chapter_result["perf"]
summary = {
    "functional_pass": verdict["functional_pass"],
    "performance_valid": verdict["performance_valid"],
    "performance_beneficial": verdict["performance_beneficial"],
    "mean_a": perf["path_a"]["mean_tokens_per_s"],
    "cv_a_percent": perf["path_a"]["cv_percent"],
    "mean_b": perf["path_b"]["mean_tokens_per_s"],
    "cv_b_percent": perf["path_b"]["cv_percent"],
    "delta_percent_b_vs_a": perf["delta_percent_b_vs_a"],
    "comparison_scope": chapter_manifest["comparison_scope"],
}
assert summary["functional_pass"] is True
assert summary["performance_valid"] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 完成标准

客观题正确；简单题形成基线/设备/资产预检；中等题产生当前输出目录的 manifest、原始日志与 result；困难题分别给出功能、数据有效性和性能收益结论，并明确本实验比较的是复合实现包。Path B 变慢仍可通过功能与数据验收。


In [ ]:
# 完成四类考核后按需执行；Notebook 不会自动展开答案。
!cat answer/02.03_answer.md
